# Walk-forward folds and per-fold clustering

This notebook sets up the evaluation design before any model is fit. It mirrors `folds/build_folds.py`: the functions below are copied from that script, so both produce identical output.

1. **Folds:** split the year into expanding training windows, each followed by a test month, plus a final held-out period.
2. **Imputation:** learn fill values from each fold's training rows only.
3. **Clustering:** group suppliers by reliability using only what was known at each training end, and compare k-means, Gaussian mixture and hierarchical clustering.
4. **Load a fold:** rebuild any fold's ready-to-model train and test sets.

**Input:** `data/processed/<scenario>/features.parquet`
**Output:** `data/processed/<scenario>/folds/` (fold definitions, cluster labels, clustering diagnostics)

> Columns starting with `true_` are the simulator's hidden truth. Here they are used **only** to report how well clustering recovered the real groups, never to make a choice.

## Setup

In [3]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from build_features import supplier_snapshot

PROCESSED = ROOT / "data" / "processed"
SCENARIO = "clusters"   # "clusters", "no_clusters" or "drift"


In [4]:
SEED = 20261028


BURN_IN_MONTHS = 1


FOLD_TEST_MONTHS = (6, 7, 8, 9)


HOLDOUT_MONTHS = (10, 12)


EMBARGO = pd.Timedelta(hours=96)


K_RANGE = range(2, 8)


N_BOOTSTRAP = 20


MIN_STABILITY = 0.75


MIN_SNAPSHOT_SHIPMENTS = 20


ALGORITHMS = ("kmeans", "gmm", "hierarchical")


CLUSTER_FEATURES = ["late_rate", "mean_late_delay", "log_delay_sd"]

In [5]:
features = pd.read_parquet(PROCESSED / SCENARIO / "features.parquet")
roles = json.loads((PROCESSED / SCENARIO / "feature_columns.json").read_text())
numeric = [c for c in roles["delay_features"] + roles["shortage_features"]
           if pd.api.types.is_numeric_dtype(features[c]) and not pd.api.types.is_bool_dtype(features[c])]
start = features["production_date"].min().normalize()
print(f"{len(features):,} rows from {start.date()}; {len(numeric)} numeric model features")

645,927 rows from 2025-01-06; 30 numeric model features


## Step 1 · Walk-forward folds

Time-series data can't be split at random. A model would then train on September to predict July, which never happens in practice. Instead each fold trains on everything up to a date and tests on the month after:

| Fold | Train on months | Test on month | Purpose |
|---|---|---|---|
| fold_1 – fold_4 | 1 → 5, 6, 7, 8 (expanding) | 6, 7, 8, 9 | compare models and choose |
| holdout | 1 → 9 | 10 – 12 | final report, used **once** |

- **Month 0 is burn-in:** its deliveries only provide history for features. They have almost no history of their own, so they aren't used as training rows.
- **96-hour embargo:** a truck due just before the training end could still be up to 96 hours late, so its outcome wouldn't be known at training time. Rows in the last 96 hours before each training end are dropped from training.

In [6]:
def make_folds(start: pd.Timestamp) -> list[dict]:
    month = lambda m: start + pd.DateOffset(months=m)
    folds = []
    for i, m in enumerate(FOLD_TEST_MONTHS, start=1):
        folds.append(dict(fold_id=f"fold_{i}", train_start=month(BURN_IN_MONTHS),
                          train_end=month(m), test_start=month(m), test_end=month(m + 1)))
    folds.append(dict(fold_id="holdout", train_start=month(BURN_IN_MONTHS),
                      train_end=month(HOLDOUT_MONTHS[0]),
                      test_start=month(HOLDOUT_MONTHS[0]), test_end=month(HOLDOUT_MONTHS[1])))
    return folds


def split(features: pd.DataFrame, fold: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    t = features["planned_delivery_datetime"]
    train = features[(t >= fold["train_start"]) & (t < fold["train_end"] - EMBARGO)]
    test = features[(t >= fold["test_start"]) & (t < fold["test_end"])]
    return train, test

In [7]:
folds = make_folds(start)
summary = []
for f in folds:
    tr, te = split(features, f)
    summary.append({"fold": f["fold_id"], "train_from": f["train_start"].date(), "train_to": f["train_end"].date(),
                    "test_from": f["test_start"].date(), "test_to": f["test_end"].date(),
                    "train_rows": len(tr), "test_rows": len(te)})
pd.DataFrame(summary)

,fold,train_from,train_to,test_from,test_to,train_rows,test_rows
0,fold_1,2025-02-06,2025-07-06,2025-07-06,2025-08-06,269505,33114
1,fold_2,2025-02-06,2025-08-06,2025-08-06,2025-09-06,306934,62420
2,fold_3,2025-02-06,2025-09-06,2025-09-06,2025-10-06,363882,55272
3,fold_4,2025-02-06,2025-10-06,2025-10-06,2025-11-06,424594,63432
4,holdout,2025-02-06,2025-11-06,2025-11-06,2026-01-06,485306,93415


fold_1's test month is smaller because it includes a plant's two-week summer shutdown.

## Step 2 · Imputation learned on training rows only

If fill values were computed on the full year, test-period information would leak into training. So each fold learns its own medians from its training rows. A few gaps have a meaning that a median would get wrong, so they're filled by rule first:

- **Never late so far:** days since last late shipment becomes the length of the supplier's history, not a typical value.
- **No shipments in the last 30 days:** fall back to the 90-day rate, then the all-history rate.

In [8]:
def fit_imputer(train: pd.DataFrame, numeric_cols: list[str]) -> dict:
    medians = train[numeric_cols].median(numeric_only=True)
    return {c: float(v) for c, v in medians.items() if pd.notna(v)}


def apply_imputer(df: pd.DataFrame, medians: dict) -> pd.DataFrame:
    df = df.copy()
    # Never late so far -> days since late = length of history (not the median)
    if "sup_days_since_late" in df:
        df["sup_days_since_late"] = df["sup_days_since_late"].fillna(df["sup_history_days"])
    # No shipments in a window -> no delay evidence; fall back to longer windows
    for short, long in [("sup_late_rate_30d", "sup_late_rate_90d"),
                        ("sup_late_rate_90d", "sup_late_rate_all"),
                        ("sup_mean_delay_30d", "sup_mean_delay_90d"),
                        ("sup_mean_delay_90d", "sup_mean_delay_all")]:
        if short in df and long in df:
            df[short] = df[short].fillna(df[long])
    df["sup_late_rate_trend"] = df["sup_late_rate_30d"] - df["sup_late_rate_90d"]
    for col, value in medians.items():
        if col in df:
            df[col] = df[col].astype("float").fillna(value)
    return df

In [9]:
fold = folds[0]
train, test = split(features, fold)
medians = fit_imputer(train, numeric)
before = train[numeric].isna().sum()
after = apply_imputer(train, medians)[numeric].isna().sum()
pd.DataFrame({"missing_before": before, "missing_after": after}).query("missing_before > 0")

,missing_before,missing_after
distance_km,12183,0
sup_late_rate_all,462,0
sup_mean_delay_all,462,0
sup_mean_late_delay_all,1919,0
sup_log_delay_sd_all,4966,0
sup_late_rate_30d,462,0
sup_mean_delay_30d,462,0
sup_late_rate_90d,462,0
sup_mean_delay_90d,462,0
sup_mean_late_delay_90d,2299,0


## Step 3 · Supplier clustering at the training end

For each fold, take a **snapshot** of every supplier using only shipments known by the training end. Then cluster suppliers on three features that describe *how* they are unreliable:

- **late rate** (logit scale): how often late
- **mean delay when late** (log scale): how late
- **spread of log delay:** how unpredictable

Suppliers with fewer than 20 known shipments aren't clustered. They're labelled `NEW` (cold start). The features are standardized so each one counts equally.

In [10]:
def clustering_matrix(snapshot: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Eligible suppliers and their standardized clustering features."""
    s = snapshot[snapshot["n_shipments"] >= MIN_SNAPSHOT_SHIPMENTS].copy()
    x = pd.DataFrame({
        "late_rate": np.log(s["late_rate"].clip(0.005, 0.995) / (1 - s["late_rate"].clip(0.005, 0.995))),
        "mean_late_delay": np.log(s["mean_late_delay"]),
        "log_delay_sd": s["log_delay_sd"],
    }, index=s.index)
    x = x.fillna(x.median())
    x = (x - x.mean()) / x.std(ddof=0)
    return s, x.to_numpy()

In [11]:
snapshot = supplier_snapshot(features, fold["train_end"])
eligible, x = clustering_matrix(snapshot)
print(f"{len(snapshot)} suppliers in snapshot, {len(eligible)} with enough history to cluster")
eligible[["supplier_code", "n_shipments"] + CLUSTER_FEATURES].describe().round(3)

356 suppliers in snapshot, 355 with enough history to cluster


,n_shipments,late_rate,mean_late_delay,log_delay_sd
count,355.000,355.000,352.000,350.000
mean,547.090,0.123,4.964,0.613
std,344.986,0.122,4.038,0.275
min,25.000,0.000,1.390,0.148
25%,290.000,0.032,2.227,0.414
50%,398.000,0.057,3.002,0.508
75%,664.000,0.200,7.242,0.759
max,1588.000,0.572,19.840,1.559


### Three algorithms, scored three ways

- **k-means:** the simple baseline; assumes round, similar-sized groups.
- **Gaussian mixture (GMM):** allows stretched groups, gives membership probabilities and has a BIC for choosing k.
- **Hierarchical (Ward):** builds groups bottom-up; a check that the structure doesn't depend on one method. It has no `predict`, so new points go to the nearest group centre.

For every algorithm and k from 2 to 7:

- **Silhouette:** how well separated the groups are (higher is better).
- **BIC:** GMM only, fit penalized by complexity (lower is better).
- **Bootstrap stability:** refit on 20 resamples of suppliers and measure agreement with the original labels (adjusted Rand index; 1 = identical). Groups that change with every resample aren't real.
- **true_ari:** agreement with the simulator's true clusters. **Diagnostic only.** A real plant wouldn't have these labels, so they can't be used to choose.

In [12]:
def _fit_predict(algorithm: str, k: int, x_fit: np.ndarray, x_all: np.ndarray, seed: int):
    """Fit on x_fit, return (labels for x_all, fitted model)."""
    if algorithm == "kmeans":
        model = KMeans(k, n_init=10, random_state=seed).fit(x_fit)
        return model.predict(x_all), model
    if algorithm == "gmm":
        model = GaussianMixture(k, n_init=3, covariance_type="full", random_state=seed).fit(x_fit)
        return model.predict(x_all), model
    model = AgglomerativeClustering(k, linkage="ward").fit(x_fit)
    centroids = np.vstack([x_fit[model.labels_ == j].mean(axis=0) for j in range(k)])
    labels = ((x_all[:, None, :] - centroids[None]) ** 2).sum(-1).argmin(1)
    return labels, model


def stability(algorithm: str, k: int, x: np.ndarray, reference: np.ndarray, rng) -> float:
    """Mean ARI between the full-data labels and labels from bootstrap refits."""
    scores = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, len(x), len(x))
        labels, _ = _fit_predict(algorithm, k, x[idx], x, int(rng.integers(1e9)))
        scores.append(adjusted_rand_score(reference, labels))
    return float(np.mean(scores))


def true_clusters_at(snapshot_suppliers: pd.DataFrame, features: pd.DataFrame,
                     as_of: pd.Timestamp, start: pd.Timestamp) -> pd.Series:
    """Simulator truth at `as_of`, for diagnostics only."""
    truth = features.drop_duplicates("supplier_code").set_index("supplier_code")[
        ["true_cluster", "true_cluster_after_drift", "true_drift_month"]]
    switch = truth["true_drift_month"].map(lambda m: start + pd.DateOffset(months=int(m)) if pd.notna(m) else pd.NaT)
    current = truth["true_cluster"].where(~(switch <= as_of).fillna(False), truth["true_cluster_after_drift"])
    return current.reindex(snapshot_suppliers["supplier_code"]).reset_index(drop=True)


def cluster_fold(features: pd.DataFrame, fold: dict, start: pd.Timestamp, rng):
    """Fit all algorithms for all k on one fold's training snapshot."""
    as_of = fold["train_end"]
    snapshot = supplier_snapshot(features, as_of)
    eligible, x = clustering_matrix(snapshot)
    truth = true_clusters_at(eligible, features, as_of, start)
    has_truth = truth.notna().all()

    rows, labels = [], {}
    for algorithm in ALGORITHMS:
        for k in K_RANGE:
            lab, model = _fit_predict(algorithm, k, x, x, SEED)
            labels[(algorithm, k)] = lab
            rows.append(dict(
                fold_id=fold["fold_id"], algorithm=algorithm, k=k, n_suppliers=len(x),
                silhouette=silhouette_score(x, lab),
                bic=model.bic(x) if algorithm == "gmm" else np.nan,
                stability_ari=stability(algorithm, k, x, lab, rng),
                true_ari=adjusted_rand_score(truth.astype(str), lab) if has_truth else np.nan,
            ))
    diag = pd.DataFrame(rows)
    return snapshot, eligible, diag, labels

In [13]:
rng = np.random.default_rng(SEED)
snapshot, eligible, diag, labels = cluster_fold(features, fold, start, rng)
diag.pivot_table(index="k", columns="algorithm",
                 values=["silhouette", "stability_ari", "bic", "true_ari"]).round(3)

c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\A

bic silhouette                     stability_ari                     true_ari                    
algorithm       gmm        gmm hierarchical kmeans           gmm hierarchical kmeans      gmm hierarchical kmeans
k                                                                                                                
2          2103.760      0.510        0.569  0.569         0.565        1.000  1.000    0.708        0.548  0.548
3          1837.064      0.595        0.595  0.595         1.000        1.000  1.000    0.921        0.921  0.921
4          1772.579      0.609        0.609  0.608         0.999        0.998  0.999    1.000        0.997  0.993
5          1767.802      0.490        0.590  0.419         0.862        0.829  0.738    0.891        0.967  0.664
6          1817.561      0.424        0.421  0.401         0.661        0.690  0.751    0.672        0.697  0.634
7          1860.359      0.413        0.389  0.415         0.679        0.691  0.780    0.873        0.518  0.583

### Choosing k, and naming clusters consistently

The rule is set **before** looking at results:

1. Keep values of k whose stability is at least 0.75. If none qualify, keep all.
2. k-means and hierarchical: highest silhouette. GMM: lowest BIC.

Cluster numbers are arbitrary, and "cluster 0" in one fold can be a different group in the next. So clusters are renamed by late rate: **C1 is always the most reliable**.

In [14]:
def choose_k(diag: pd.DataFrame) -> pd.DataFrame:
    """Pre-set rule: among stable k, best silhouette (BIC for GMM)."""
    picks = []
    for (fold_id, algorithm), g in diag.groupby(["fold_id", "algorithm"]):
        stable = g[g["stability_ari"] >= MIN_STABILITY]
        g = stable if len(stable) else g
        best = g.loc[g["bic"].idxmin()] if algorithm == "gmm" else g.loc[g["silhouette"].idxmax()]
        picks.append(best)
    chosen = pd.DataFrame(picks)[["fold_id", "algorithm", "k"]].assign(chosen=True)
    return diag.merge(chosen, on=["fold_id", "algorithm", "k"], how="left").fillna({"chosen": False})


def ordered_labels(raw: np.ndarray, late_rate: pd.Series) -> np.ndarray:
    """Rename clusters C1..Ck by mean late rate, so C1 is always the most reliable."""
    order = pd.Series(late_rate.to_numpy()).groupby(raw).mean().sort_values().index
    names = {old: f"C{new + 1}" for new, old in enumerate(order)}
    return np.array([names[v] for v in raw])

In [15]:
diag = choose_k(diag)
chosen = diag[diag["chosen"]][["algorithm", "k", "silhouette", "stability_ari", "true_ari"]]
display(chosen.round(3))

algorithm = "kmeans"
k = int(chosen.set_index("algorithm").loc[algorithm, "k"])
profile = eligible.assign(cluster=ordered_labels(labels[(algorithm, k)], eligible["late_rate"]))
print(f"{algorithm}, k={k}: cluster profiles")
profile.groupby("cluster")[["late_rate", "mean_late_delay", "log_delay_sd"]].agg(["mean", "size"]).round(3)

,algorithm,k,silhouette,stability_ari,true_ari
2,kmeans,4,0.608,0.999,0.993
9,gmm,5,0.490,0.862,0.891
14,hierarchical,4,0.609,0.998,0.997


kmeans, k=4: cluster profiles


late_rate      mean_late_delay      log_delay_sd     
             mean size            mean size         mean size
cluster                                                      
C1          0.034  186           2.325  186        0.490  186
C2          0.113   64           9.192   64        1.090   64
C3          0.248   68           3.672   68        0.401   68
C4          0.355   37          13.075   37        0.783   37

## Step 4 · All folds, and which algorithm feeds models 2 and 4

`run` repeats Steps 1–3 for every fold and saves the results. The algorithm used by models 2 and 4 is then chosen by another pre-set rule: **highest mean stability at its chosen k across the four CV folds**, ties broken by silhouette. Only training-side diagnostics are used, so neither the holdout nor `true_ari` influences the choice. (Takes about 40 seconds.)

In [16]:
def select_algorithm(diag: pd.DataFrame) -> str:
    """Pre-set rule on training-only diagnostics across the CV folds (not the holdout)."""
    chosen = diag[diag["chosen"] & (diag["fold_id"] != "holdout")]
    summary = chosen.groupby("algorithm")[["stability_ari", "silhouette"]].mean()
    return summary.sort_values(["stability_ari", "silhouette"], ascending=False).index[0]


def run(scenario: str) -> None:
    print(f"[folds] scenario={scenario}")
    features = pd.read_parquet(PROCESSED / scenario / "features.parquet")
    roles = json.loads((PROCESSED / scenario / "feature_columns.json").read_text())
    numeric = [c for c in roles["delay_features"] + roles["shortage_features"]
               if pd.api.types.is_numeric_dtype(features[c]) and not pd.api.types.is_bool_dtype(features[c])]

    start = features["production_date"].min().normalize()
    folds = make_folds(start)
    rng = np.random.default_rng(SEED)

    all_diag, all_labels = [], []
    for fold in folds:
        train, test = split(features, fold)
        assert train["planned_delivery_datetime"].max() < fold["train_end"] - EMBARGO
        assert test["planned_delivery_datetime"].min() >= fold["train_end"]
        fold["imputation_medians"] = fit_imputer(train, numeric)
        fold["n_train_rows"], fold["n_test_rows"] = len(train), len(test)

        snapshot, eligible, diag, labels = cluster_fold(features, fold, start, rng)
        diag = choose_k(diag)
        all_diag.append(diag)

        lab = snapshot[["supplier_code", "n_shipments"]].copy()
        lab.insert(0, "fold_id", fold["fold_id"])
        fold["chosen_k"] = {}
        for algorithm in ALGORITHMS:
            k = int(diag[(diag["algorithm"] == algorithm) & diag["chosen"]]["k"].iloc[0])
            fold["chosen_k"][algorithm] = k
            named = pd.Series(ordered_labels(labels[(algorithm, k)], eligible["late_rate"]),
                              index=eligible.index)
            lab[f"cluster_{algorithm}"] = named.reindex(lab.index).fillna("NEW")
        all_labels.append(lab)
        print(f"  {fold['fold_id']}: train {len(train):,} rows, test {len(test):,} rows, "
              f"{len(eligible)} suppliers clustered, k={fold['chosen_k']}")

    diag = pd.concat(all_diag, ignore_index=True)
    selected = select_algorithm(diag)
    labels = pd.concat(all_labels, ignore_index=True)
    labels["cluster"] = labels[f"cluster_{selected}"]
    print(f"  selected algorithm for models 2 and 4: {selected}")

    out = PROCESSED / scenario / "folds"
    out.mkdir(parents=True, exist_ok=True)
    diag.to_csv(out / "clustering_diagnostics.csv", index=False)
    labels.to_parquet(out / "cluster_labels.parquet", index=False)
    serial = [{k: (str(v) if isinstance(v, pd.Timestamp) else v) for k, v in f.items()} for f in folds]
    (out / "fold_definitions.json").write_text(json.dumps(
        {"selected_algorithm": selected, "embargo_hours": EMBARGO.total_seconds() / 3600,
         "folds": serial}, indent=2))
    print(f"  wrote {out}")

In [17]:
run(SCENARIO)
all_diag = pd.read_csv(PROCESSED / SCENARIO / "folds" / "clustering_diagnostics.csv")
all_diag[all_diag["chosen"]].pivot_table(index="fold_id", columns="algorithm",
                                         values=["k", "stability_ari", "true_ari"]).round(2)

[folds] scenario=clusters


c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\A

  fold_1: train 269,505 rows, test 33,114 rows, 355 suppliers clustered, k={'kmeans': 4, 'gmm': 5, 'hierarchical': 4}


c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\A

  fold_2: train 306,934 rows, test 62,420 rows, 358 suppliers clustered, k={'kmeans': 4, 'gmm': 4, 'hierarchical': 4}


...

c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\A

  fold_4: train 424,594 rows, test 63,432 rows, 366 suppliers clustered, k={'kmeans': 4, 'gmm': 4, 'hierarchical': 4}


c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\AppData\Local\anaconda3\envs\supplier-risk-engine\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\leer1\A

  holdout: train 485,306 rows, test 93,415 rows, 366 suppliers clustered, k={'kmeans': 4, 'gmm': 4, 'hierarchical': 4}
  selected algorithm for models 2 and 4: hierarchical
  wrote C:\Users\leer1\Documents\GitHub\bayesian-supplier-risk-engine-jit\data\processed\clusters\folds


k                     stability_ari                     true_ari                    
algorithm  gmm hierarchical kmeans           gmm hierarchical kmeans      gmm hierarchical kmeans
fold_id                                                                                          
fold_1     5.0          4.0    4.0          0.86          1.0   1.00     0.89         1.00   0.99
fold_2     4.0          4.0    4.0          1.00          1.0   1.00     1.00         1.00   0.99
fold_3     4.0          4.0    4.0          1.00          1.0   0.99     1.00         0.99   0.99
fold_4     4.0          4.0    4.0          1.00          1.0   1.00     1.00         1.00   1.00
holdout    4.0          4.0    4.0          1.00          1.0   1.00     1.00         1.00   1.00

**What to expect across scenarios** (rerun with a different `SCENARIO` to see):

- **clusters:** every algorithm finds k = 4 with stability and true_ari near 1.0.
- **no_clusters:** only k = 2 survives, with low stability for GMM and hierarchical. The algorithms are cutting a continuum in half, not finding real groups.
- **drift:** true_ari falls to about 0.5–0.8. The snapshot summarizes each supplier's *whole* history, so a supplier that switched recently is a blend of its old and new behaviour. A snapshot built from a recent window only would track switches faster but be noisier. That trade-off is worth testing later.

## Step 5 · Load a ready-to-model fold

`load_fold` rebuilds one fold's train and test sets from the saved artifacts: split by date, impute with that fold's training medians, and attach that fold's cluster labels. Suppliers that weren't clustered get `NEW`.

In [18]:
def load_fold(scenario: str, fold_id: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Rebuild one fold's train and test sets: split, impute, attach cluster labels."""
    folder = PROCESSED / scenario / "folds"
    meta = json.loads((folder / "fold_definitions.json").read_text())
    fold = next(f for f in meta["folds"] if f["fold_id"] == fold_id)
    fold = {k: (pd.Timestamp(v) if k.endswith(("_start", "_end")) else v) for k, v in fold.items()}

    features = pd.read_parquet(PROCESSED / scenario / "features.parquet")
    labels = pd.read_parquet(folder / "cluster_labels.parquet")
    labels = labels[labels["fold_id"] == fold_id].drop(columns=["fold_id", "n_shipments"])

    out = []
    for part in split(features, fold):
        part = apply_imputer(part, fold["imputation_medians"])
        part = part.merge(labels, on="supplier_code", how="left")
        cluster_cols = [c for c in part.columns if c.startswith("cluster")]
        part[cluster_cols] = part[cluster_cols].fillna("NEW")   # not yet in the snapshot
        out.append(part)
    return out[0], out[1]

In [19]:
train, test = load_fold(SCENARIO, "fold_2")
model_cols = [c for c in numeric if c in train]
print(f"train {train.shape}, test {test.shape}")
print("missing values left in model features:", int(train[model_cols].isna().sum().sum()),
      "train /", int(test[model_cols].isna().sum().sum()), "test")
test.groupby("cluster").agg(suppliers=("supplier_code", "nunique"),
                            late_rate=("is_late", "mean"),
                            mean_delay_hours=("delay_hours", "mean")).round(3)

train (306934, 64), test (62420, 64)
missing values left in model features: 0 train / 0 test


,suppliers,late_rate,mean_delay_hours
cluster,,,
C1,187,0.033,0.076
C2,66,0.105,0.886
C3,68,0.241,0.869
C4,37,0.334,4.422
NEW,8,0.124,0.594


## Next steps

1. **Scoring code and naive baseline:** for each fold, predict every test delivery's delay from the supplier's trailing average, and compute CRPS. This tests the fold loop end to end before any slow model is added.
2. **Model 1:** Bayesian hurdle model (P(late) × log-normal delay) on the delay features.
3. **Model 2:** model 1 plus the `cluster` label.
4. **Models 3 and 4:** add supplier-level effects, then suppliers nested within clusters.